# 3.2 Logistic Regression

**Core series, class 7.** Table 1 (3.1) showed that cases are older, heavier, and have different blood pressure and cholesterol. But those differences overlap: older people are also heavier, on average. **Logistic regression** asks how each characteristic goes with the **odds of type 2 diabetes** while **holding the others constant**.

### By the end of this notebook you'll be able to

1. Explain **odds** and **odds ratios**, and why logistic regression is used for yes/no outcomes
2. Fit **univariable** and **multivariable** models with `glm()`, and read them with `broom::tidy()`
3. Put predictors on **useful scales** (per 10 years, per 5 BMI units)
4. Compare models fitted on the **same people**
5. Explain why some variables (HbA1c, and LDL after treatment) mislead, and what **reverse causation** means

### 🔎 Analysis at a glance

| | |
|---|---|
| ❓ **Question** | Which characteristics go with higher odds of type 2 diabetes, accounting for each other? |
| 🧪 **The analysis** | **Logistic regression**: type 2 diabetes (yes/no) ~ age + sex + BMI + blood pressure + LDL, built up step by step |
| 🎯 **Why** | The outcome is yes/no, so logistic regression; **multivariable**, because the predictors are related to each other (confounding) |
| ✅ **What we conclude** | Adjusted odds ratios: e.g. how much higher the odds are per 5 BMI units, at the same age and sex |
| ⚠️ **What it can't tell us** | Cause. Measurements taken **after** diagnosis can reflect the disease or its treatment (**reverse causation**). And our case-control sample gives odds ratios, not risks |
| 🔗 **Connections** | Uses `final_analysis.csv`; the same kind of model as the GWAS in **4.2**; **3.3** is the version for a numeric outcome |

| | |
|---|---|
| 📂 **Uses** | `final_analysis.csv` (from **2.3**) |
| 💾 **Saves** | nothing |

### Class plan

| Part | Minutes |
|---|---|
| 1. Key concepts | 20 |
| 2. Load and prepare | 10 |
| 3. One predictor at a time | 15 |
| 4. Multivariable models | 20 |
| 5. What the model can't tell us | 15 |
| Wrap-up | 5 |

### ⏱️ Time and 💰 cost

| | |
|---|---|
| ⏱️ **Class time** | 85 minutes |
| ⏱️ **Run time** | under **1 minute** to run every cell |
| 💰 **Data cost** | **$0**: no BigQuery queries |
| 💰 **App cost** | about **$0.27 an hour** while the app is running. The app's disk costs about **$0.70 a day**, even while the app is stopped |
| **Estimated total** | about **$0.40** per student for this class (app time only) |

> ⚠️ **Keep All of Us data in the Workbench.** Don't share counts of 1 to 20. Clear outputs before you share or commit this notebook (**Edit → Clear Outputs of All Cells**).

## Part 1. Key concepts

### Odds and odds ratios

- **Odds** = (chance it happens) ÷ (chance it doesn't). A 20% chance is odds of 0.2 ÷ 0.8 = **0.25** ("1 to 4").
- **Odds ratio (OR)** = the odds in one group ÷ the odds in another. **OR = 1**: no difference; **OR = 2**: twice the odds; **OR = 0.5**: half the odds.
- For a **number** like BMI, the OR is **per unit**: OR 1.4 "per 5 kg/m²" means each 5 extra BMI units go with 1.4 times the odds.

### Why logistic regression?

The outcome is **yes/no** (type 2 diabetes or not). Ordinary (linear) regression could predict impossible values, like a 120% chance. **Logistic regression** models the **log odds**, which can be any number, and gives us odds ratios: `exp(coefficient)`.

### Why odds ratios suit our sample

Our sample is **half cases**, by design (a **case-control** sample). That makes the **risk** (percent with diabetes) meaningless here, but the **odds ratio** is still valid: it's the same whether you sample by outcome or not. That's why case-control studies report odds ratios.

### Univariable vs multivariable

| | Univariable | Multivariable |
|---|---|---|
| Model | diabetes ~ **one** predictor | diabetes ~ **several** predictors |
| Answers | is this predictor related to diabetes at all? | is it related **at the same values of the others**? |
| OR is | **crude** (unadjusted) | **adjusted** |

If an OR shrinks a lot after adjustment, part of the crude link was due to the other variables: **confounding**.

### Functions

| Code | What it does |
|---|---|
| `glm(y ~ x1 + x2, data, family = binomial)` | fits a logistic regression |
| `broom::tidy(model, exponentiate = TRUE, conf.int = TRUE)` | a table of ORs with 95% confidence intervals |
| `reformulate("x", "y")` | builds the formula `y ~ x` from text |
| `drop_na(a, b, c)` | keeps rows with no missing values **in these columns** |

## Part 2. Load and prepare

Three steps:

1. **Outcome as 0/1:** `glm()` needs *No* = 0, *Yes* = 1.
2. **Useful scales:** an OR "per 1 year" or "per 1 mg/dL" is too small to read. We divide: **per 10 years** of age, **per 5** BMI units, **per 10** mm Hg, **per 10** mg/dL of LDL.
3. **Sex at birth:** we keep *Female* and *Male*; other answers are too few for stable estimates (that's a limitation to report, not a judgment).

In [ ]:
source("../aou_setup.R")  # settings, packages, and helper functions
library(broom)

fa <- prepare_analysis_data() %>%
  mutate(t2d_binary = if_else(t2d == "Yes", 1, 0),
         age10 = age / 10,
         BMI5  = BMI / 5,
         SBP10 = SBP / 10,
         LDL10 = LDL / 10) %>%
  filter(sex_at_birth %in% c("Female", "Male")) %>%
  mutate(sex_at_birth = droplevels(sex_at_birth))

fa %>% count(t2d, t2d_binary)

**🔍 Read the output:** every *No* is 0 and every *Yes* is 1, with both groups well represented.

## Part 3. One predictor at a time

`univariable()` fits a model with one predictor and returns its OR. Each model uses everyone who has that predictor, so the number of people (`n`) differs between rows.

In [ ]:
univariable <- function(predictor) {
  model <- glm(reformulate(predictor, "t2d_binary"), data = fa, family = binomial)
  tidy(model, exponentiate = TRUE, conf.int = TRUE) %>%
    filter(term != "(Intercept)") %>%
    mutate(n = nobs(model))
}

map(c("age10", "sex_at_birth", "BMI5", "SBP10", "LDL10"), univariable) %>%
  list_rbind() %>%
  select(term, OR = estimate, conf.low, conf.high, p.value, n) %>%
  mutate(across(c(OR, conf.low, conf.high), ~ round(.x, 2)))

**🔍 Read the output:**

| Column | Means |
|---|---|
| `term` | the predictor (`sex_at_birthMale` = male compared with female) |
| `OR` | the **crude** odds ratio, per 10 years, per 5 BMI units, etc. |
| `conf.low`, `conf.high` | the 95% confidence interval. If it doesn't include 1, the link is unlikely to be chance |
| `n` | people with this predictor |

**Age** and **BMI** have strong ORs above 1 (in our test run: 1.43 per 10 years, 1.46 per 5 BMI units). Look at **LDL**: its OR is **below 1** (0.87 per 10 mg/dL in our test run). Part 5 explains why.

## Part 4. Multivariable models

To **compare** models fairly, they must use the **same people**. Otherwise a change in an OR could just be a change in who's in the model. So we first make one data set with every model variable present (`drop_na()` on just these columns), and report its size.

In [ ]:
model_data <- fa %>% drop_na(t2d_binary, age10, sex_at_birth, BMI5, SBP10, LDL10)
nrow(model_data)
model_data %>% count(t2d)

models <- list(
  "1. Age + sex"         = glm(t2d_binary ~ age10 + sex_at_birth,                       data = model_data, family = binomial),
  "2. + BMI"             = glm(t2d_binary ~ age10 + sex_at_birth + BMI5,                data = model_data, family = binomial),
  "3. + blood pressure, LDL" = glm(t2d_binary ~ age10 + sex_at_birth + BMI5 + SBP10 + LDL10, data = model_data, family = binomial)
)

imap(models, ~ tidy(.x, exponentiate = TRUE, conf.int = TRUE) %>% mutate(model = .y)) %>%
  list_rbind() %>%
  filter(term != "(Intercept)") %>%
  select(model, term, OR = estimate, conf.low, conf.high, p.value) %>%
  mutate(across(c(OR, conf.low, conf.high), ~ round(.x, 2))) %>%
  arrange(term, model)

**🔍 Read the output:** each predictor appears once per model it's in, so you can follow its OR as the model grows.

- **Fewer people:** requiring LDL removes many, mostly **controls** (LDL is measured more often in people with diabetes; 3.1). Check `count(t2d)`: the model data is no longer half and half.
- **BMI** gets **stronger** after adjusting for age (1.46 crude to 1.61 in model 2, in our test run). Adjustment can move an OR either way: here, younger people are heavier on average but have less diabetes, which hid part of BMI's link.
- **Blood pressure** has a modest OR once BMI is in the model (1.09 per 10 mm Hg in our test run).
- In our test run the models used **6,397** people, out of 9,905: mostly controls without an LDL result.
- `imap()` is like `map()`, but also passes each list element's **name** (`.y`), which becomes the `model` column.

### Comparing models

**AIC** (Akaike information criterion) balances fit against complexity: **lower is better**, and only comparable for models fitted on the same data (which is why we fixed `model_data`).

In [ ]:
imap(models, ~ glance(.x) %>% mutate(model = .y)) %>%
  list_rbind() %>%
  select(model, nobs, AIC)

**🔍 Read the output:** AIC drops as useful predictors are added. The biggest drop usually comes from **BMI**.

## Part 5. What the model can't tell us

### Why not HbA1c?

HbA1c is one of the tests used to **diagnose** type 2 diabetes. Putting it in a model "predicting" diabetes is circular: it would show a huge OR that tells us nothing new, and it would distort the other ORs. **Don't adjust for a variable that defines the outcome.**

### Why is LDL's OR below 1? Reverse causation

High LDL cholesterol raises the risk of heart disease, and people with diabetes are at high heart risk, so guidelines recommend **statins** for most adults with diabetes. Our LDL value is the **latest** one (2.2), usually measured **after** diagnosis and treatment. So low LDL in cases is partly a **result** of having diabetes, not a cause of it: **reverse causation**.

The fix isn't a better model, it's better **timing**: use measurements from **before** the diagnosis (a baseline).

**💬 Discuss:** which other predictors here could be affected by having diabetes, or by its treatment? (Weight can change with diabetes medicines; blood pressure is often treated too.)

### ✍️ Your turn

Requiring LDL dropped many people. Refit model 2 (**age + sex + BMI**) on everyone with those three variables. How many people does it use, and does the BMI OR change?

<details>
<summary><b>Show answer</b></summary>

```r
bigger <- fa %>% drop_na(t2d_binary, age10, sex_at_birth, BMI5)
glm(t2d_binary ~ age10 + sex_at_birth + BMI5, data = bigger, family = binomial) %>%
  tidy(exponentiate = TRUE, conf.int = TRUE) %>%
  filter(term == "BMI5") %>%
  mutate(n = nrow(bigger))
```

Many more people, especially controls, and a BMI OR that's usually similar. This is a **sensitivity analysis**: if a result holds with different, reasonable choices (here, who's included), it's more convincing.

</details>

In [ ]:
# Your code here

## ⚠️ Common errors

| Message | Means | Fix |
|---|---|---|
| *glm.fit: fitted probabilities numerically 0 or 1 occurred* | A predictor almost perfectly separates the groups (e.g. HbA1c) | Leave it out, or check the variable |
| ORs like 1.002 that look like "no effect" | The predictor is on a tiny scale (per 1 mg/dL) | Rescale (per 10 mg/dL) |
| Models with different `nobs` | Each model dropped different missing rows | Fit all models on one `model_data` |
| *contrasts can be applied only to factors with 2 or more levels* | A factor has one level left after filtering | `droplevels()`, or check the filter |

## ✅ Key takeaways

- Logistic regression is for **yes/no** outcomes; `exp(coefficient)` is the **odds ratio**
- **Adjusted** ORs hold the other predictors constant; compare models on the **same people**
- Put predictors on **readable scales**
- Don't adjust for what **defines** the outcome (HbA1c)
- Measurements taken **after** diagnosis can show **reverse causation** (LDL and statins)

**Next:** **3.3 linear_regression**: the same idea for a **numeric** outcome, systolic blood pressure.

**Before you share or commit this notebook:** clear all outputs (**Edit → Clear Outputs of All Cells**).